# STAI-X 2026 — Model Training & CV Analysis

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_train, load_val
from src.features import FeaturePipeline
from src.models import train_all, CATEGORIES

sns.set_theme(style='whitegrid')

In [ ]:
train = load_train()
val   = load_val()
pipeline = FeaturePipeline()
train_feat, val_feat = pipeline.fit_transform(train, val)
print('Ready. Feature count:', len(pipeline.feature_cols))

## Train all models (GroupKFold by period_rank)

In [ ]:
results = train_all(train_feat, pipeline.feature_cols, verbose=True)

## OOF MAE summary table

In [ ]:
rows = []
for cat in CATEGORIES:
    r = results[cat]
    rows.append({'category': cat, **r['mae']})
summary = pd.DataFrame(rows).set_index('category')
print('OOF MAE per category and model:')
print(summary.to_string())
print(f"\nCompetition metric (mean ensemble MAE): {summary['ensemble'].mean():.4f}")

## OOF prediction scatter (ensemble)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, cat in zip(axes, CATEGORIES):
    r = results[cat]
    mask = (train_feat['overdose_category'] == cat) & train_feat['rate_per_10000_ed_visits'].notna()
    y_true = train_feat.loc[mask, 'rate_per_10000_ed_visits'].values
    y_pred = r['oof_ensemble']
    ax.scatter(y_true, y_pred, alpha=0.4, s=15)
    lim = max(y_true.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', lw=1)
    ax.set_title(f"{cat}\nMAE={r['mae']['ensemble']:.3f}")
    ax.set_xlabel('True')
    ax.set_ylabel('Predicted')
plt.tight_layout()
plt.show()

## LightGBM feature importance (first fold, each category)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, cat in zip(axes, CATEGORIES):
    m = results[cat]['lgb_models'][0]
    imp = pd.Series(m.feature_importances_, index=pipeline.feature_cols)
    top = imp.nlargest(15)
    top.sort_values().plot(kind='barh', ax=ax)
    ax.set_title(f'LGB importance — {cat}')
plt.tight_layout()
plt.show()

## Stimulants deep-dive (hardest category)

In [ ]:
cat = 'all_stimulants'
r = results[cat]
mask = (train_feat['overdose_category'] == cat) & train_feat['rate_per_10000_ed_visits'].notna()
df_stim = train_feat[mask].copy()
df_stim['oof_pred'] = r['oof_ensemble']
df_stim['abs_error'] = (df_stim['rate_per_10000_ed_visits'] - df_stim['oof_pred']).abs()

print('Worst 10 predictions (by abs error):')
worst = df_stim.nlargest(10, 'abs_error')[['jurisdiction','period_id','period_rank',
                                            'rate_per_10000_ed_visits','oof_pred','abs_error']]
print(worst.to_string(index=False))